In [32]:
"""
NBA Games Dtataet: Machine Learning Analysis
Models: Decision Tree and SVM (Support Vector Machine)
Target: HOME_TEAM_WINS (1 = home team won, 0 = away team won)

Dataset source: https://www.kaggle.com/datasets/nathanlauga/nba-games
"""


'\nNBA Games Dtataet: Machine Learning Analysis\nModels: Decision Tree and SVM (Support Vector Machine)\nTarget: HOME_TEAM_WINS (1 = home team won, 0 = away team won)\n\nDataset source: https://www.kaggle.com/datasets/nathanlauga/nba-games\n'

In [33]:
# importing libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)

## Step 1: Loading the data

In [34]:
# loading data 
df = pd.read_csv('nba_games.csv')

## Step 2: Analysing and preprocessing the data

In [35]:
# inspecting data
df.shape

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25024 entries, 0 to 25023
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   GAME_DATE_EST     25024 non-null  object 
 1   GAME_ID           25024 non-null  int64  
 2   GAME_STATUS_TEXT  25024 non-null  object 
 3   HOME_TEAM_ID      25024 non-null  int64  
 4   VISITOR_TEAM_ID   25024 non-null  int64  
 5   SEASON            25024 non-null  int64  
 6   TEAM_ID_home      25024 non-null  int64  
 7   PTS_home          24925 non-null  float64
 8   FG_PCT_home       24925 non-null  float64
 9   FT_PCT_home       24925 non-null  float64
 10  FG3_PCT_home      24925 non-null  float64
 11  AST_home          24925 non-null  float64
 12  REB_home          24925 non-null  float64
 13  TEAM_ID_away      25024 non-null  int64  
 14  PTS_away          24925 non-null  float64
 15  FG_PCT_away       24925 non-null  float64
 16  FT_PCT_away       24925 non-null  float6

In [36]:
df.head()

,GAME_DATE_EST,GAME_ID,GAME_STATUS_TEXT,HOME_TEAM_ID,VISITOR_TEAM_ID,SEASON,TEAM_ID_home,PTS_home,FG_PCT_home,FT_PCT_home,...,AST_home,REB_home,TEAM_ID_away,PTS_away,FG_PCT_away,FT_PCT_away,FG3_PCT_away,AST_away,REB_away,HOME_TEAM_WINS
0,2021-11-17,22100213,Final,1610612766,1610612764,2021,1610612766,97.0,0.438,0.500,...,30.0,59.0,1610612764,87.0,0.367,0.813,0.190,23.0,48.0,1
1,2021-11-17,22100214,Final,1610612765,1610612754,2021,1610612765,97.0,0.425,0.750,...,16.0,42.0,1610612754,89.0,0.418,0.737,0.243,14.0,43.0,1
2,2021-11-17,22100215,Final,1610612737,1610612738,2021,1610612737,110.0,0.506,0.833,...,28.0,40.0,1610612738,99.0,0.440,0.824,0.268,24.0,42.0,1
3,2021-11-17,22100216,Final,1610612751,1610612739,2021,1610612751,109.0,0.458,0.840,...,29.0,47.0,1610612739,99.0,0.393,0.857,0.250,20.0,50.0,1
4,2021-11-17,22100217,Final,1610612748,1610612740,2021,1610612748,113.0,0.483,0.824,...,29.0,39.0,1610612740,98.0,0.440,0.786,0.286,18.0,38.0,1


In [37]:
# Dropping ID columns as they carry no predictive value and the model would just memorise the IDs and not generalise well 
df = df.drop(columns=['GAME_DATE_EST', 'GAME_ID', 'GAME_STATUS_TEXT', 'HOME_TEAM_ID', 'VISITOR_TEAM_ID',
                        'SEASON', 'TEAM_ID_home', 'TEAM_ID_away'])

In [38]:
# Confirming the change
df.head()

,PTS_home,FG_PCT_home,FT_PCT_home,FG3_PCT_home,AST_home,REB_home,PTS_away,FG_PCT_away,FT_PCT_away,FG3_PCT_away,AST_away,REB_away,HOME_TEAM_WINS
0,97.0,0.438,0.500,0.313,30.0,59.0,87.0,0.367,0.813,0.190,23.0,48.0,1
1,97.0,0.425,0.750,0.286,16.0,42.0,89.0,0.418,0.737,0.243,14.0,43.0,1
2,110.0,0.506,0.833,0.351,28.0,40.0,99.0,0.440,0.824,0.268,24.0,42.0,1
3,109.0,0.458,0.840,0.375,29.0,47.0,99.0,0.393,0.857,0.250,20.0,50.0,1
4,113.0,0.483,0.824,0.375,29.0,39.0,98.0,0.440,0.786,0.286,18.0,38.0,1


In [39]:
# checking for null values and dropping them as only 99 of 25,024 rows (~0.4%) are empty,
# small enough that dropping is safer than imputing.
df = df.dropna()

In [40]:
# Dropping 'PTS_home' and 'PTS_away' columns as well as our target column is 'HOME_TEAM_WINS'.
# If we include those columns, data leakage would occur and the model would learn the label definition. 
df = df.drop(columns=['PTS_home', 'PTS_away'])

In [41]:
# Final dataset shape before model training
print("Final dataset shape:", df.shape)
print("Class balance:\n", df['HOME_TEAM_WINS'].value_counts(normalize=True))

Final dataset shape: (24925, 11)
Class balance:
 HOME_TEAM_WINS
1    0.591214
0    0.408786
Name: proportion, dtype: float64


## Step 3: Test/train split

In [42]:
# Train/test split
X = df.drop(columns=['HOME_TEAM_WINS'])
y = df['HOME_TEAM_WINS']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("\nTrain shape:", X_train.shape, "Test shape:", X_test.shape)


Train shape: (19940, 10) Test shape: (4985, 10)


## Step 4: Decision Tree Model

In [43]:
# Model 1: Decision Tree
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)
y_proba_dt = dt.predict_proba(X_test)[:, 1]

print("Decision Tree Results")
print("Accuracy:  %.3f" % accuracy_score(y_test, y_pred_dt))
print("Precision: %.3f" % precision_score(y_test, y_pred_dt))
print("Recall:    %.3f" % recall_score(y_test, y_pred_dt))
print("F1 score:  %.3f" % f1_score(y_test, y_pred_dt))
print("ROC-AUC:   %.3f" % roc_auc_score(y_test, y_proba_dt))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_dt))

Decision Tree Results
Accuracy:  0.790
Precision: 0.794
Recall:    0.870
F1 score:  0.830
ROC-AUC:   0.868

Confusion Matrix:
 [[1373  665]
 [ 384 2563]]


In [44]:
# checking which features played major roles in the decision tree
importances = pd.Series(dt.feature_importances_, index=X_train.columns) \
                .sort_values(ascending=False)
print("\nTop 5 most important features (Decision Tree):")
print(importances.head(5))


Top 5 most important features (Decision Tree):
FG_PCT_away     0.481806
FG_PCT_home     0.451592
REB_home        0.032193
REB_away        0.023994
FG3_PCT_home    0.010414
dtype: float64


## Step 5: SVM Model

In [45]:
# Model 2: SVM 
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)  # fit on train only, avoid data leakage

svm = SVC(kernel='rbf', probability=True, random_state=42)
svm.fit(X_train_scaled, y_train)

y_pred_svm = svm.predict(X_test_scaled)
y_proba_svm = svm.predict_proba(X_test_scaled)[:, 1]

print("\nSVM Results")
print("Accuracy: %.3f" % accuracy_score(y_test, y_pred_svm))
print("Precision: %.3f" % precision_score(y_test, y_pred_svm))
print("Recall: %.3f" % recall_score(y_test, y_pred_svm))
print("F1 score: %.3f" % f1_score(y_test, y_pred_svm))
print("ROC-AUC: %.3f" % roc_auc_score(y_test, y_proba_svm))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_svm))


SVM Results
Accuracy: 0.831
Precision: 0.838
Recall: 0.886
F1 score: 0.861
ROC-AUC: 0.911

Confusion Matrix:
 [[1535  503]
 [ 337 2610]]
